# 00 Data Check

Purpose: verify that the research environment is alive before RL experiments, FinRL baselines or application integration begin.

This notebook checks:

- environment variables
- local and project paths
- guldNAS canonical paths when mounted
- DuckDB connection
- raw API response folders
- Parquet and CSV folders
- minimal market data availability

This notebook must run without full RL training and without requiring unavailable API keys.


## 1. Imports and helper functions


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv


def path_status(path: Path) -> dict:
    """Return a small status record for a filesystem path."""
    return {
        "path": str(path),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "is_file": path.is_file(),
    }


def count_files(path: Path, pattern: str = "*") -> int:
    """Count files matching a pattern. Returns 0 if the folder does not exist."""
    if not path.exists() or not path.is_dir():
        return 0
    return sum(1 for item in path.glob(pattern) if item.is_file())


def resolve_path(raw_path: str | None, base_path: Path) -> Path | None:
    """Resolve an environment path relative to a base path unless it is absolute."""
    if not raw_path:
        return None
    path = Path(raw_path).expanduser()
    if path.is_absolute():
        return path
    return (base_path / path).resolve()


## 2. Environment variables


In [2]:
# This notebook is expected to run from the project root or from research/notebooks.
# The paths below resolve the repository structure based on this notebook location convention.

current_working_dir = Path.cwd().resolve()

if current_working_dir.name == "notebooks":
    research_dir = current_working_dir.parent
    project_root = research_dir.parent
elif current_working_dir.name == "research":
    research_dir = current_working_dir
    project_root = research_dir.parent
else:
    project_root = current_working_dir
    research_dir = project_root / "research"

system_dir = project_root / "system"
research_env_path = research_dir / ".env"
system_env_path = system_dir / ".env"

load_dotenv(research_env_path)

env_values = {
    "RESEARCH_ENV": os.getenv("RESEARCH_ENV"),
    "LOG_LEVEL": os.getenv("LOG_LEVEL"),
    "DUCKDB_PATH": os.getenv("DUCKDB_PATH"),
    "GULDNAS_DUCKDB_PATH": os.getenv("GULDNAS_DUCKDB_PATH"),
    "SYSTEM_RUNTIME_DATA_PATH": os.getenv("SYSTEM_RUNTIME_DATA_PATH"),
    "RESEARCH_RESULTS_PATH": os.getenv("RESEARCH_RESULTS_PATH"),
    "RESEARCH_FIGURES_PATH": os.getenv("RESEARCH_FIGURES_PATH"),
    "RESEARCH_TABLES_PATH": os.getenv("RESEARCH_TABLES_PATH"),
}

pd.DataFrame(env_values.items(), columns=["variable", "value"])


,variable,value
0,RESEARCH_ENV,local
1,LOG_LEVEL,INFO
2,DUCKDB_PATH,../system/runtime-data/market_research.duckdb
3,GULDNAS_DUCKDB_PATH,/mnt/nas/stockinvestmentdss/duckdb/market_rese...
4,SYSTEM_RUNTIME_DATA_PATH,../system/runtime-data
5,RESEARCH_RESULTS_PATH,./results
6,RESEARCH_FIGURES_PATH,./results/figures
7,RESEARCH_TABLES_PATH,./results/tables


## 3. Local path checks


In [3]:
local_paths = {
    "project_root": project_root,
    "research_dir": research_dir,
    "system_dir": system_dir,
    "research_env": research_env_path,
    "system_env": system_env_path,
    "system_runtime_data": system_dir / "runtime-data",
    "research_results": research_dir / "results",
    "research_tables": research_dir / "results" / "tables",
    "research_figures": research_dir / "results" / "figures",
    "research_logs": research_dir / "results" / "logs",
    "research_models": research_dir / "results" / "models",
}

pd.DataFrame([{"name": name, **path_status(path)} for name, path in local_paths.items()])


,name,path,exists,is_dir,is_file
0,project_root,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,True,False
1,research_dir,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,True,False
2,system_dir,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,True,False
3,research_env,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,False,True
4,system_env,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,False,True
5,system_runtime_data,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,True,True,False
6,research_results,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False
7,research_tables,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False
8,research_figures,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False
9,research_logs,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False


## 4. guldNAS path checks


In [4]:
guldnas_duckdb_path = Path(os.getenv("GULDNAS_DUCKDB_PATH", "/mnt/nas/stockinvestmentdss/duckdb/market_research.duckdb"))
guldnas_paths = {
    "guldnas_duckdb_file": guldnas_duckdb_path,
    "guldnas_duckdb_folder": guldnas_duckdb_path.parent,
    "guldnas_project_root": Path("/mnt/nas/stockinvestmentdss"),
}

pd.DataFrame([{"name": name, **path_status(path)} for name, path in guldnas_paths.items()])


,name,path,exists,is_dir,is_file
0,guldnas_duckdb_file,\mnt\nas\stockinvestmentdss\duckdb\market_rese...,False,False,False
1,guldnas_duckdb_folder,\mnt\nas\stockinvestmentdss\duckdb,False,False,False
2,guldnas_project_root,\mnt\nas\stockinvestmentdss,False,False,False


## 5. DuckDB connection check


In [5]:
# Research .env paths are resolved relative to /research.
duckdb_path = resolve_path(os.getenv("DUCKDB_PATH"), research_dir)

if duckdb_path is None:
    duckdb_path = system_dir / "runtime-data" / "market_research.duckdb"

duckdb_path.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(duckdb_path)) as con:
    con.execute("CREATE TABLE IF NOT EXISTS notebook_connection_check (id INTEGER, status VARCHAR)")
    con.execute("DELETE FROM notebook_connection_check")
    con.execute("INSERT INTO notebook_connection_check VALUES (1, 'ok')")
    status = con.execute("SELECT status FROM notebook_connection_check WHERE id = 1").fetchone()[0]

pd.DataFrame([
    {"check": "duckdb_path", "value": str(duckdb_path)},
    {"check": "duckdb_connection", "value": status},
])


,check,value
0,duckdb_path,C:\Users\gurug\Dropbox\DataScience\Speciale\De...
1,duckdb_connection,ok


## 6. Raw API response folder check


In [6]:
raw_api_candidates = {
    "data_raw_api_responses": project_root / "data" / "raw-api-responses",
    "data_raw": project_root / "data" / "raw",
    "system_runtime_raw_api_responses": system_dir / "runtime-data" / "raw-api-responses",
}

pd.DataFrame([
    {
        "name": name,
        **path_status(path),
        "file_count": count_files(path),
    }
    for name, path in raw_api_candidates.items()
])


,name,path,exists,is_dir,is_file,file_count
0,data_raw_api_responses,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
1,data_raw,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
2,system_runtime_raw_api_responses,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0


## 7. Parquet folder check


In [7]:
parquet_candidates = {
    "data_parquet": project_root / "data" / "parquet",
    "data_parquet_raw": project_root / "data" / "parquet" / "raw",
    "data_parquet_curated": project_root / "data" / "parquet" / "curated",
    "data_parquet_features": project_root / "data" / "parquet" / "features",
}

pd.DataFrame([
    {
        "name": name,
        **path_status(path),
        "parquet_files": count_files(path, "*.parquet"),
    }
    for name, path in parquet_candidates.items()
])


,name,path,exists,is_dir,is_file,parquet_files
0,data_parquet,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
1,data_parquet_raw,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
2,data_parquet_curated,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
3,data_parquet_features,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0


## 8. CSV folder check


In [8]:
csv_candidates = {
    "data_csv": project_root / "data" / "csv",
    "data_csv_raw": project_root / "data" / "csv" / "raw",
    "data_csv_curated": project_root / "data" / "csv" / "curated",
    "data_csv_features": project_root / "data" / "csv" / "features",
}

pd.DataFrame([
    {
        "name": name,
        **path_status(path),
        "csv_files": count_files(path, "*.csv"),
    }
    for name, path in csv_candidates.items()
])


,name,path,exists,is_dir,is_file,csv_files
0,data_csv,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
1,data_csv_raw,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
2,data_csv_curated,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0
3,data_csv_features,C:\Users\gurug\Dropbox\DataScience\Speciale\De...,False,False,False,0


## 9. Minimal market data availability table


In [9]:
market_data_checks = []

candidate_files = []
for folder in list(parquet_candidates.values()) + list(csv_candidates.values()):
    if folder.exists() and folder.is_dir():
        candidate_files.extend(folder.glob("*.parquet"))
        candidate_files.extend(folder.glob("*.csv"))

market_data_checks.append({
    "source": "local_parquet_csv_files",
    "available": len(candidate_files) > 0,
    "count": len(candidate_files),
    "note": "Counts local Parquet/CSV files in expected data folders.",
})

with duckdb.connect(str(duckdb_path)) as con:
    tables = con.execute("SHOW TABLES").fetchdf()

market_data_checks.append({
    "source": "duckdb_tables",
    "available": len(tables) > 0,
    "count": len(tables),
    "note": "Counts tables currently present in the configured DuckDB file.",
})

pd.DataFrame(market_data_checks)


,source,available,count,note
0,local_parquet_csv_files,False,0,Counts local Parquet/CSV files in expected dat...
1,duckdb_tables,True,2,Counts tables currently present in the configu...


## 10. Optional FinRL import check


In [10]:
optional_imports = []

for package_name in ["finrl", "torch", "stable_baselines3", "gymnasium"]:
    try:
        __import__(package_name)
        optional_imports.append({"package": package_name, "import_status": "OK"})
    except Exception as exc:
        optional_imports.append({"package": package_name, "import_status": f"FAILED: {type(exc).__name__}: {exc}"})

pd.DataFrame(optional_imports)


,package,import_status
0,finrl,OK
1,torch,OK
2,stable_baselines3,OK
3,gymnasium,OK


## 11. Summary / next actions


In [11]:
summary = [
    {"item": "DuckDB connection", "status": "OK" if status == "ok" else "CHECK"},
    {"item": "Local fallback DuckDB path", "status": str(duckdb_path)},
    {"item": "guldNAS DuckDB path exists", "status": guldnas_duckdb_path.exists()},
    {"item": "Local market data files found", "status": len(candidate_files) > 0},
    {"item": "DuckDB tables found", "status": len(tables) > 0},
]

pd.DataFrame(summary)


,item,status
0,DuckDB connection,OK
1,Local fallback DuckDB path,C:\Users\gurug\Dropbox\DataScience\Speciale\De...
2,guldNAS DuckDB path exists,False
3,Local market data files found,False
4,DuckDB tables found,True
